# Demo: rebuild the submission from the stored weights

This notebook reproduces our selected submission for the Solar Filament Segmentation
Challenge 2026 **without any training**. It uses only the pixels of each test image.

Pipeline: 10 U-Nets (ResNet34 encoder; 5 folds at 1024 px and 5 folds at 2048 px) give a mean
probability map; the map is split into candidate filaments; 5 seed refiners re-segment each
candidate; a learned filter (LightGBM) removes candidates that annotators are unlikely to mark;
the result is one mask per filament in COCO RLE.

Our two selected Kaggle submissions come from this notebook: `USE_FILTER = True` gives the
first (`mix10_refiner5_filter1.csv`, public 0.38), `USE_FILTER = False` the second
(`mix10_refiner5.csv`, public 0.37).

**Runtime.** On a GPU (RTX 3070) all 180 test images take 5 to 10 minutes (1.5 s per image).
On a CPU the pipeline needs about 80 s per image, so start with `N_IMAGES = 3`.

**Exactness.** float16 results differ slightly between GPU generations, so the CSV is
byte-identical to the submitted file (same sha256) only on the GPU type that made it: the
filtered file on an NVIDIA Quadro RTX 8000, the unfiltered file on an NVIDIA RTX 3070. On other
GPUs and on a CPU the masks agree with a mean IoU above 0.997.

**Setup** (from the repository root):

```bash
uv venv --python 3.12 .venv && source .venv/bin/activate
uv pip install -r requirements.txt
```

Put the competition test images in `data/MAGFiLO_1.0_Kaggle_2026/test/test_images/`. The
notebook downloads the weights into `release/weights/` on the first run and checks their SHA-256.

## 1. Settings

In [ ]:
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TEST_IMAGES = REPO / "data" / "MAGFiLO_1.0_Kaggle_2026" / "test" / "test_images"
WEIGHTS = REPO / "release" / "weights"
# The weights are downloaded once when WEIGHTS is missing (release asset of this repository).
WEIGHTS_URL = "https://github.com/tokyovo/solar-filament-segmentation-2026/releases/download/v1.0/filament-seg-weights.tar"
WEIGHTS_SHA256 = "ec3d8df80ac7e48662f5cd5d472a80f8c8565ea5060b748f13349e38d6069c55"
N_IMAGES = 3          # None = all 180 test images
OUT = REPO / "submission_demo.csv"
USE_FILTER = True    # True: selected submission 1; False: selected submission 2
# Expected sha256 of each submitted file, and the GPU type that made it.
EXPECTED = {True: ("9fac0e35011809beeac63a4a9045ce0e73b8a603a30d7dab64745c195a862f72", "Quadro RTX 8000"),
            False: ("e66c87e65679f14bd1eb8790a98f23a5809fb02a7cf011281329eeb1ee0a082e", "GeForce RTX 3070")}

## 2. Load the code and check the device

In [ ]:
import hashlib, sys
sys.path.insert(0, str(REPO))
import torch
from src import predict
from src.weights import fetch_weights

if not WEIGHTS.exists():
    print("downloading the weights (about 700 MiB) ...")
    fetch_weights(WEIGHTS_URL, WEIGHTS_SHA256, WEIGHTS)

print("device:", predict.DEVICE, "| torch", torch.__version__)
first_stage = [WEIGHTS / f"unet_r34_f{f}" for f in range(5)] + [WEIGHTS / f"unet_r34_r2048_f{f}" for f in range(5)]
refiners = [WEIGHTS / f"refiner_v2_f{f}" for f in range(5)]
filter_dir = WEIGHTS / "filter_v1" if USE_FILTER else None
missing = [str(p) for p in first_stage + refiners if not (p / "model.pt").exists()]
if USE_FILTER and not (filter_dir / "model.txt").exists():
    missing.append(str(filter_dir))
assert not missing, f"missing weights: {missing}"
paths = sorted(TEST_IMAGES.glob("*.jpeg"))
assert paths, f"no test images in {TEST_IMAGES}"
paths = paths if N_IMAGES is None else paths[:N_IMAGES]
print(f"{len(paths)} images, {len(first_stage)} first-stage models, {len(refiners)} refiners")

## 3. Predict and write the CSV

In [ ]:
import time

start = time.time()
frame = predict.predict_frame(paths, first_stage, refiners, threshold=0.5, min_area=300, merge_dist=15,
                              filter_dir=filter_dir)
frame.to_csv(OUT, index=False)
print(f"{len(frame)} filaments in {len(paths)} images, {time.time() - start:.0f} s -> {OUT}")
frame.head()

## 4. Compare with the submitted file (full run only)

In [ ]:
sha = hashlib.sha256(OUT.read_bytes()).hexdigest()
expected, gpu = EXPECTED[USE_FILTER]
if N_IMAGES is None:
    print("sha256", sha)
    print("identical to the submitted file" if sha == expected
          else f"differs from the submitted file; exact equality needs an NVIDIA {gpu} (float16 differs between GPU types)")
else:
    print("partial run: the sha256 check needs N_IMAGES = None")

## 5. Look at one image

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
from src import data

name = paths[0].stem
image = data.load_image(paths[0])
overlay = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
for counts in frame.loc[frame["filament_id"].str.startswith(name + "_"), "segmentation_rle"]:
    mask = data.rle_string_to_mask(counts)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(overlay, contours, -1, (255, 60, 0), 3)
plt.figure(figsize=(9, 9))
plt.imshow(overlay)
plt.title(f"{name}: predicted filaments")
plt.axis("off")
plt.show()